# Return State Discovery 02 — Volatility states → conditional price behavior

**Status: HISTORICAL EXPLORATORY.** Infer anonymous volatility states with an HDP-HSMM **without any signed-return emissions**. Then test whether *filtered, pre-block* volatility state probabilities improve the **next non-overlapping 30-minute return density**, signed-return momentum/reversion, and direction scores beyond AM-T-HAR-Blend and continuous volatility conditioning.

Frozen research specification: [`RSD 002`](../../discussions/Return_State_Discovery/002_chatgpt_volatility_states_price_conditioning_preregistration.md). The historical validation/test data have already been inspected in VD14–18 and RSD01: reported intervals are exploratory and **not** prospective confirmation. The finite weak-limit Bayesian HDP prior does not prove a discrete regime count. Only aggregate artifacts will be saved.

## 0. Environment, existing price history, and isolated synthetic tests

In [ ]:
import os,sys,json,time,subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(),'Launch Jupyter from the cloned repo'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as v14
from data_engineering.modeling import markov_har_variance as v18
from data_engineering.modeling import hdp_hsmm_return_states as rsd
from data_engineering.modeling import hdp_hsmm_volatility_price as m
cfg=m.Registration();assert cfg.kcap==12 and cfg.horizon_min==30
HIST=Path(os.environ['DERIBIT_DATA_ROOT'])/'history'/'price_5m'
assert HIST.is_dir(),f'Missing Binance 5m history: {HIST}'
OUT=ROOT/'notebooks'/'Return_State_Discovery'/'02';OUT.mkdir(parents=True,exist_ok=True)
assets=cfg.primary_assets
START=time.monotonic()
p=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_hdp_hsmm_volatility_price.py')],cwd=ROOT,capture_output=True,text=True)
print(p.stdout[-2500:]);print(p.stderr[-1000:]);assert p.returncode==0
print('Frozen RSD02:',json.dumps(m.registration(),indent=2))

## 1. Refit pre-test VD18 and build observed-volatility emissions

VD18 AM-T-HAR-Blend is fitted **only before** the original 150-day historical TEST. Its forecast at the start of each 30-minute block is frozen before the six subsequent 5m returns are observed. `rv30` and its derived state emissions become known only at the end of that block. The realized-variance floor must be fitted on RSD02 train alone.

In [ ]:
raw,excluded=v14.load_returns(HIST,tuple(assets))
panel=v14.panelize(raw);del raw
features=v14.make_features(panel)
frame=v14.make_targets(panel,features,v18.HORIZONS)
frame=v18.add_har_lags(frame)
assert set(frame.underlying.unique())==set(assets)
base,baseline_contract=rsd.build_historical_30m_blocks(frame,particles=256,cal_limit=512)
base=m.attach_realized_variance(frame,base)
base=base.sort_values(['underlying','state_time']).reset_index(drop=True)
all_days=sorted(pd.to_datetime(base.state_time,utc=True).dt.floor('D').unique())
cut1,cut2=all_days[int(len(all_days)*.6)],all_days[int(len(all_days)*.8)]
base['period']=np.where(base.state_time<cut1,'fit',np.where(base.state_time<cut2,'diagnostic','evaluation'))
blocks=[];floors={}
for asset,g in base.groupby('underlying',sort=True):
    tr=g.loc[g.period=='fit'];pos=tr.rv30.to_numpy();pos=pos[pos>0]
    floor=max(float(np.median(pos)*.001) if len(pos) else 1e-12,1e-16)
    out,_=m.build_volatility_features(g,floor=floor)
    out['underlying']=asset
    floors[asset]=floor
    blocks.append(out)
vol=pd.concat(blocks,ignore_index=True)
vol=vol.loc[vol.vol_ready].copy()
assert np.isfinite(vol[['vol_1','vol_2','vol_3','return_surprise']]).all().all()
print('VD18 pretest contract:',baseline_contract)
print('RSD02 UTC split boundaries:',cut1,cut2,'from',len(all_days),'previously inspected TEST days')
print('RV floors (FIT only):',floors)
print(vol.groupby(['underlying','period']).agg(blocks=('R_30','size'),days=('state_time',lambda x:x.dt.floor('D').nunique())))
print('Elapsed minutes',round((time.monotonic()-START)/60,1))

## 2. Fit HDP-HSMM to **volatility observations only**

Two independent 140-iteration chains per asset, Student-t(6) emissions, Kcap=12, explicit-duration age cap 96 half-hour blocks (not compulsory switching). The number of occupied states is a **posterior statistic, not a proven number of economic regimes**. The state model never receives signed returns or future price-path efficiency.

In [ ]:
def contiguous(g,min_len):
    q=g.sort_values('state_time').reset_index(drop=True)
    seg=rsd.contiguous_segments(q.state_time,min_length=min_len)
    assert seg,f'No segments with >={min_len} blocks'
    ix=np.concatenate([np.arange(a,b) for a,b in seg])
    out=q.iloc[ix].reset_index(drop=True)
    bounds=np.r_[0,np.cumsum([b-a for a,b in seg])].astype(int)
    return out,bounds

state_data={};chain_rows=[];profile_rows=[]
sp=rsd.Spec(kcap=cfg.kcap,dmax=cfg.age_cap_blocks,n_iter=cfg.n_iter,burn=cfg.burn,
            thin=cfg.thin,df=cfg.df,seed=cfg.hsmm_chains[0])
for asset in assets:
    g=vol.loc[vol.underlying==asset]
    train,tb=contiguous(g.loc[g.period=='fit'],24)
    evaluate,eb=contiguous(g.loc[g.period=='evaluation'],1)
    diagnostic,db=contiguous(g.loc[g.period=='diagnostic'],1)
    norm=rsd.fit_normalizer(train[['vol_1','vol_2','vol_3']].to_numpy())
    X=rsd.apply_normalizer(train[['vol_1','vol_2','vol_3']].to_numpy(),norm)
    chains=[]
    for ci,seed in enumerate(cfg.hsmm_chains,1):
        ss=rsd.Spec(**{**sp.__dict__,'seed':seed})
        post=rsd.fit_posterior(X,ss,segments=tb)
        summ=rsd.posterior_summary(post)
        chain_rows.append({'asset':asset,'chain':ci,**{k:v for k,v in summ.items() if k!='K_histogram'},
                           'K_histogram':json.dumps(summ['K_histogram'])})
        print(asset,'chain',ci,'occupied K:',summ['K_histogram'],'MH:',round(summ['beta_mh_acceptance'],3),'cap:',summ['warning'])
        rep=post['draws'][len(post['draws'])//2]
        for k in np.flatnonzero(rep['occupancy']>0):
            episodes=rep['episode_lengths'][rep['episode_states']==k]
            profile_rows.append({'asset':asset,'chain':ci,'state_internal_id':int(k),
                 'occupancy_blocks':int(rep['occupancy'][k]),
                 'emission_logvar_z':float(rep['mu'][k,0]),
                 'emission_RVsurprise_z':float(rep['mu'][k,1]),
                 'emission_fastslow_z':float(rep['mu'][k,2]),
                 'episode_count':int(len(episodes)),
                 'median_episode_blocks':float(np.median(episodes)) if len(episodes) else np.nan})
        chains.append(post)
    state_data[asset]={'train':train,'train_bounds':tb,'eval':evaluate,'eval_bounds':eb,
                       'diag':diagnostic,'diag_bounds':db,'norm':norm,'chains':chains}
posterior_chains=pd.DataFrame(chain_rows)
vol_states=pd.DataFrame(profile_rows)
display(posterior_chains)
display(vol_states)
print('These are unsigned-volatility-state descriptions, not return regime names.')

## 3. Next-block predictive RETURN scores on strictly causal state probabilities

Every block's return is scored using only the state probabilities from **earlier completed blocks**. It is only after the block ends that its observed squared returns update the filter. State→price response parameters are fitted on earlier RSD02 FIT blocks, independently by posterior draw, and predictive densities averaged across draws/chains. Controls: unchanged VD18, recent Student-t, and continuous-volatility Student-t allowing causal heteroskedasticity and return dependence.

In [ ]:
all_scores=[];all_boot=[];all_price_profiles=[];eval_cache={};eval_behavior=[]
for asset in assets:
    z=state_data[asset]
    sc,boot,profs,perblock=m.run_asset(z['train'],z['eval'],z['chains'],z['norm'],
                        z['train_bounds'],z['eval_bounds'],cfg.posterior_draws_per_chain,cfg.n_bootstrap)
    all_scores.append({'asset':asset,'eval_blocks':len(perblock),'eval_days':perblock.date.nunique(),**sc})
    all_boot.append(boot.assign(asset=asset))
    all_price_profiles.extend([{'asset':asset,**p} for p in profs])
    eval_cache[asset]=perblock
    for chain_ix,posterior in enumerate(z['chains'],start=1):
        e=m.future_price_profiles(z['eval'],posterior,z['norm'],z['eval_bounds'],chain=chain_ix)
        e['asset']=asset
        eval_behavior.append(e)
model_scores=pd.DataFrame(all_scores);bootstrap=pd.concat(all_boot,ignore_index=True)
price_profiles=pd.DataFrame(all_price_profiles)
future_behavior=pd.concat(eval_behavior,ignore_index=True)
display(model_scores.T)
display(bootstrap)
print('Next-block behavior conditional on state probabilities known BEFORE each block:')
display(future_behavior.loc[future_behavior.effective_future_blocks>=100].round(4))
print('Positive HDP_VOL_STATE minus CONTINUOUS_T NLL = no added predictive return value.')

## 4. Sign-randomized return negative control

Flip an **entire completed 30m return block's sign**, independently across blocks. RV and inferred VOL states stay exactly unchanged, while directional predictability is destroyed. Refit state→return and continuous return-response parameters on randomly flipped TRAIN, evaluate on independently flipped EVALUATION. This is a *directional null*, not a null of volatility-state existence.

In [ ]:
sign_rows=[]
for asset in assets:
    z=state_data[asset]
    q=m.sign_randomization(z['train'],z['eval'],z['chains'],z['norm'],z['train_bounds'],z['eval_bounds'],
                            B=cfg.n_sign_null,seed=2901 if asset=='BTC' else 2907,
                            draws_per_chain=cfg.posterior_draws_per_chain)
    q['asset']=asset;sign_rows.append(q)
sign_controls=pd.concat(sign_rows,ignore_index=True)
for asset in assets:
    actual=bootstrap.loc[(bootstrap.asset==asset)&(bootstrap.a=='HDP_VOL_STATE_sign_brier')&
                         (bootstrap.b=='CONTINUOUS_T_sign_brier'),'delta'].iloc[0]
    q=sign_controls.loc[sign_controls.asset==asset,'state_minus_cont_sign_brier']
    print(asset,'actual state minus continuous sign Brier=',round(float(actual),6),
          'sign-flip null 5%-95%:',q.quantile([.05,.95]).round(6).to_dict(),
          'fraction null <= actual:',round(float((q<=actual).mean()),3))
print('40 sign-null repetitions are exploratory; not an independently calibrated regime-test p-value.')

## 5. Conditional regime-free HEAVY-TAIL volatility null

Retain each original TRAIN block's actual **ex-ante predicted variance**, simulate six independent standardized Student-t(6) returns, calculate their squared returns, and fit the same HDP-HSMM to the reconstructed volatility emissions. Volatility changes continuously by construction: this null has **no explicit discrete states**. Because we do not re-run HAR recursively inside simulations, it is a conditional diagnostic, not a full model-comparison significance test.

In [ ]:
null_rows=[]
for asset in assets:
    z=state_data[asset];train=z['train'];norm=z['norm']
    for nullseed in (311,719):
        sim=m.fixed_variance_regime_free_null(train,seed=nullseed,df=cfg.df)
        sim,_=m.build_volatility_features(sim,floor=floors[asset])
        sim,bs=contiguous(sim.loc[sim.vol_ready],min_len=24)
        XX=rsd.apply_normalizer(sim[['vol_1','vol_2','vol_3']].to_numpy(),norm)
        ns=rsd.Spec(**{**sp.__dict__,'seed':nullseed})
        post=rsd.fit_posterior(XX,ns,segments=bs)
        diag=rsd.posterior_summary(post)
        null_rows.append({'asset':asset,'null_seed':nullseed,
                 'sim_train_blocks':len(sim),'K_occupied_median':diag['K_occupied_median'],
                 'K_histogram':json.dumps(diag['K_histogram']),
                 'at_Kcap_probability':diag['at_Kcap_probability'],
                 'duration_cap_rate':diag['at_duration_cap_mean'],
                 'beta_mh_acceptance':diag['beta_mh_acceptance'],'warning':diag['warning']})
null_counts=pd.DataFrame(null_rows);display(null_counts)
print('Posterior K under this no-regime null is not an economic state count.')

## 6. Frozen exploratory decision, concise reproducibility artifacts

Primary: state-conditioned NLL must beat **continuous-volatility Student-t** and unchanged VD18 at BTC and ETH, with negative paired day-bootstrap upper bound, before a regime-dependent return effect can be considered a candidate. This is not independent confirmation. The sign-null and variance-null are essential interpretation checks. Never name states from a single draw as market mechanisms.

In [ ]:
checks=[]
for asset in assets:
    bb=bootstrap[bootstrap.asset==asset]
    def find(a,b):return bb[(bb.a==a)&(bb.b==b)].iloc[0]
    vscont=find('HDP_VOL_STATE_nll','CONTINUOUS_T_nll')
    vshar=find('HDP_VOL_STATE_nll','AM_T_HAR_Blend_nll')
    sign=find('HDP_VOL_STATE_sign_brier','CONTINUOUS_T_sign_brier')
    chains=posterior_chains[posterior_chains.asset==asset]
    sensitive=(chains.warning!='NONE').any() or chains.K_occupied_median.max()-chains.K_occupied_median.min()>1
    qs=sign_controls[sign_controls.asset==asset].state_minus_cont_sign_brier
    checks.append({'asset':asset,'delta_NLL_state_minus_cont':float(vscont.delta),
                   'NLL_upper_day_CI_state_minus_cont':float(vscont.ci975),
                   'delta_NLL_state_minus_VD18':float(vshar.delta),
                   'NLL_upper_day_CI_state_minus_VD18':float(vshar.ci975),
                   'direction_brier_state_minus_cont':float(sign.delta),
                   'sign_null_percentile':float((qs<=sign.delta).mean()),
                   'chain_uncertainty_warning':bool(sensitive),
                   'exploratory_candidate':bool(vscont.ci975<0 and vshar.ci975<0 and not sensitive)})
checks=pd.DataFrame(checks)
verdict='EXPLORATORY_CANDIDATE' if checks.exploratory_candidate.all() else 'NO_ESTABLISHED_INCREMENTAL_PRICE_STATE'
print('Verdict:',verdict);display(checks)
for name,table in [('02_model_scores.csv',model_scores),('02_bootstrap.csv',bootstrap),
   ('02_posterior_chains.csv',posterior_chains),('02_volatility_state_profiles.csv',vol_states),
   ('02_return_response_profiles.csv',price_profiles),
   ('02_eval_conditional_behavior.csv',future_behavior),('02_sign_null.csv',sign_controls),
   ('02_regime_free_null.csv',null_counts),('02_checks.csv',checks)]:
    table.to_csv(OUT/name,index=False,float_format='%.9g')
manifest={'module':'Return_State_Discovery_02','spec':m.registration(),
    'status':'HISTORICAL_EXPLORATORY','decision':verdict,'assets':list(assets),
    'baseline':baseline_contract,'cut_fit_before':str(cut1),'eval_after':str(cut2),
    'rv_floor_train_only':floors,'checks':checks.to_dict('records'),
    'note':['Posterior K is not evidence of economic regimes.',
            'Only past completed volatility blocks enter conditional return predictions.',
            'Historical TEST reused after VD14-18 and RSD01; prospective confirmation still required.',
            'Regime-free null conditioned on actual ex-ante variance; not a complete HAR-refitted DGP.',
            'No option strategy or executable alpha claim.']}
(OUT/'02_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved 10 aggregate artifacts under',OUT,'| runtime minutes',round((time.monotonic()-START)/60,1))
print('No raw price history, full state paths or per-origin return predictions retained.')

## Interpretation guardrails

- An HDP-HSMM can occupy multiple states in a heavy-tailed no-regime world; **do not** interpret a posterior count alone as real regimes.
- Even if state-conditioned NLL improves, first inspect whether it is all **variance calibration** (scale-only state model) rather than momentum/reversal; sign Brier/logloss must add information beyond the continuous baseline and the sign-randomization null to support *directional* interpretation.
- Forecast comparisons use completed-block state filtering; *smoothed* or same-block state labels would leak the target and are prohibited.
- Compare posterior chain consistency, state occupancy and the no-regime K distribution; 15 draws/chain are not convergence proof.
- The present model remains a **horizon-conditioned** use of AM-T-HAR-Blend, not a coherent five-minute joint price/volatility HSMM.
- Repeat any promising effect on an independently frozen prospective sample before considering options costs, implied volatility, or trading rules.